This dataset was manipulated from the data_jobs dataset created by Luke Barousse containing hundreds of thousands of real-world job postings related to data analytics, data engineering, and data science roles. It was manipulated to include only data analyst jobs using the following code:

```python
data_jobs = pd.read_csv("data_jobs.csv")

data_jobs = data_jobs[data_jobs['job_title'] == 'Data Analyst'].reset_index(drop=True)

data_jobs.to_csv('data_analyst_jobs.csv', index=False)
```

This dataset includes detailed information such as job titles, salaries, employment type, company location, remote status, required skills, and education requirements. The dataset supports analysis of hiring trends, skill demand, and salary patterns across the data industry.

In [52]:
import pandas as pd

no_degree_jobs = pd.read_csv("../data/data_analyst_jobs.csv")
no_degree_jobs.head()

,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Data Analyst,Data Analyst,"Guadalajara, Jalisco, Mexico",via BeBee México,Full-time,False,Mexico,2023-01-14 13:18:07,False,False,Mexico,NaN,NaN,NaN,Hewlett Packard Enterprise,"['r', 'python', 'sql', 'nosql', 'power bi', 't...","{'analyst_tools': ['power bi', 'tableau'], 'pr..."
1,Data Analyst,Data Analyst,"Warsaw, Poland",via Praca Trabajo.org,Full-time,False,Poland,2023-10-16 13:36:54,False,False,Poland,NaN,NaN,NaN,Glovo,"['sql', 'python', 'r', 'redshift', 'pandas', '...","{'analyst_tools': ['excel', 'looker', 'tableau..."
2,Data Analyst,Data Analyst,"Des Moines, IA",via Trabajo.org,Full-time,False,"Illinois, United States",2023-11-06 13:01:22,False,True,United States,NaN,NaN,NaN,Assuredpartners,NaN,NaN
3,Data Analyst,Data Analyst,Singapore,via BeBee Singapore,Full-time,False,Singapore,2023-12-20 13:15:45,True,False,Singapore,NaN,NaN,NaN,Moovaz,['sql'],{'programming': ['sql']}
4,Data Analyst,Data Analyst,"Tampa, FL",via LinkedIn,Full-time,False,"Florida, United States",2023-01-19 13:19:45,False,False,United States,NaN,NaN,NaN,Citi,"['sql', 'python', 'unix', 'excel', 'jira']","{'analyst_tools': ['excel'], 'async': ['jira']..."


In [53]:
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 41950 entries, 0 to 41949
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_title_short        41950 non-null  str    
 1   job_title              41950 non-null  str    
 2   job_location           41861 non-null  str    
 3   job_via                41942 non-null  str    
 4   job_schedule_type      40825 non-null  str    
 5   job_work_from_home     41950 non-null  bool   
 6   search_location        41950 non-null  str    
 7   job_posted_date        41950 non-null  str    
 8   job_no_degree_mention  41950 non-null  bool   
 9   job_health_insurance   41950 non-null  bool   
 10  job_country            41950 non-null  str    
 11  salary_rate            2383 non-null   str    
 12  salary_year_avg        1309 non-null   float64
 13  salary_hour_avg        1051 non-null   float64
 14  company_name           41950 non-null  str    
 15  job_skills   

In [54]:
no_degree_jobs.isnull().sum()

job_title_short              0
job_title                    0
job_location                89
job_via                      8
job_schedule_type         1125
job_work_from_home           0
search_location              0
job_posted_date              0
job_no_degree_mention        0
job_health_insurance         0
job_country                  0
salary_rate              39567
salary_year_avg          40641
salary_hour_avg          40899
company_name                 0
job_skills                6449
job_type_skills           6449
dtype: int64

In [55]:
# Define the columns to keep
columns_to_keep = [
    'job_title', 'salary_year_avg', 'job_schedule_type', 'job_country',
    'job_work_from_home', 'search_location', 'job_skills', 'job_no_degree_mention', 'job_posted_date', 'company_name'
]

# Create new data_analyst_jobs with columns to keep
no_degree_jobs = no_degree_jobs[columns_to_keep]
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 41950 entries, 0 to 41949
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_title              41950 non-null  str    
 1   salary_year_avg        1309 non-null   float64
 2   job_schedule_type      40825 non-null  str    
 3   job_country            41950 non-null  str    
 4   job_work_from_home     41950 non-null  bool   
 5   search_location        41950 non-null  str    
 6   job_skills             35501 non-null  str    
 7   job_no_degree_mention  41950 non-null  bool   
 8   job_posted_date        41950 non-null  str    
 9   company_name           41950 non-null  str    
dtypes: bool(2), float64(1), str(7)
memory usage: 2.6 MB


In [56]:
# Renaming columns for consistency
no_degree_jobs = no_degree_jobs.rename(columns={
    'salary_year_avg': 'salary_usd',
    'job_schedule_type': 'employment_type',
    'job_country': 'company_location',
    'job_work_from_home': 'is_remote',
    'search_location': 'employee_location',
    'job_no_degree_mention': 'no_degree_mention',
    'job_posted_date': 'posting_date'
})

no_degree_jobs.columns

Index(['job_title', 'salary_usd', 'employment_type', 'company_location',
       'is_remote', 'employee_location', 'job_skills', 'no_degree_mention',
       'posting_date', 'company_name'],
      dtype='str')

In [57]:
def clean_skills(x):
    # Case 1: NaN (float)
    if isinstance(x, float):
        return []
    
    # Case 2: Real Python list
    if isinstance(x, list):
        return x
    
    # Case 3: NumPy array → convert to list
    if hasattr(x, "tolist"):
        return [skill.strip() for skill in x.tolist()]
    
    # Case 4: Stringified list like "['sql','python']"
    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        # Remove brackets
        inner = x[1:-1]
        # Split by comma
        items = inner.split(',')
        # Strip quotes and whitespace
        return [skill.strip().strip("'").strip('"') for skill in items]
    
    # Case 5: Comma-separated string
    if isinstance(x, str):
        return [skill.strip() for skill in x.split(',')]
    
    # Fallback
    return []

In [58]:
no_degree_jobs['job_skills'] = no_degree_jobs['job_skills'].apply(clean_skills)

In [59]:
no_degree_jobs['job_skills'].head()

0           [r, python, sql, nosql, power bi, tableau]
1    [sql, python, r, redshift, pandas, excel, look...
2                                                   []
3                                                [sql]
4                     [sql, python, unix, excel, jira]
Name: job_skills, dtype: object

In [ ]:
# Create indicator columns for each skill of interest
skills_to_track = ['python', 'sql', 'tableau']

for skill in skills_to_track:
    no_degree_jobs[f"has_{skill}"] = no_degree_jobs['job_skills'].apply(
        lambda skills: skill in skills
    )

# Count skill frequency for python, sql, and tableau
no_degree_jobs[['has_python', 'has_sql', 'has_tableau']].sum()

has_python     5031
has_sql        9465
has_tableau    4220
dtype: int64

In [71]:
# Count individual skills
skills_exploded = no_degree_jobs['job_skills'].explode()
skills_exploded.value_counts()

job_skills
sql            9465
excel          5588
python         5031
tableau        4220
power bi       3921
               ... 
nltk              1
fortran           1
ringcentral       1
openstack         1
rust              1
Name: count, Length: 192, dtype: int64

In [63]:
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 41950 entries, 0 to 41949
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_title          41950 non-null  str    
 1   salary_usd         1309 non-null   float64
 2   employment_type    40825 non-null  str    
 3   company_location   41950 non-null  str    
 4   is_remote          41950 non-null  bool   
 5   employee_location  41950 non-null  str    
 6   job_skills         41950 non-null  object 
 7   no_degree_mention  41950 non-null  bool   
 8   posting_date       41950 non-null  str    
 9   company_name       41950 non-null  str    
 10  has_python         41950 non-null  bool   
 11  has_sql            41950 non-null  bool   
 12  has_tableau        41950 non-null  bool   
dtypes: bool(5), float64(1), object(1), str(6)
memory usage: 2.8+ MB


In [65]:
no_degree_jobs['no_degree_mention'].value_counts()

no_degree_mention
False    23415
True     18535
Name: count, dtype: int64

In [66]:
no_degree_jobs = no_degree_jobs[
    no_degree_jobs['no_degree_mention'] == True
].copy()

In [67]:
no_degree_jobs['degree_flag'] = 0
no_degree_jobs.info()

<class 'pandas.DataFrame'>
Index: 18535 entries, 3 to 41949
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_title          18535 non-null  str    
 1   salary_usd         345 non-null    float64
 2   employment_type    18003 non-null  str    
 3   company_location   18535 non-null  str    
 4   is_remote          18535 non-null  bool   
 5   employee_location  18535 non-null  str    
 6   job_skills         18535 non-null  object 
 7   no_degree_mention  18535 non-null  bool   
 8   posting_date       18535 non-null  str    
 9   company_name       18535 non-null  str    
 10  has_python         18535 non-null  bool   
 11  has_sql            18535 non-null  bool   
 12  has_tableau        18535 non-null  bool   
 13  degree_flag        18535 non-null  int64  
dtypes: bool(5), float64(1), int64(1), object(1), str(6)
memory usage: 1.5+ MB


In [77]:
no_degree_jobs[no_degree_jobs['salary_usd'].notna()]['salary_usd'].head(10)

157     117500.0
468     100500.0
887      90000.0
968      75000.0
999      55000.0
1343     60000.0
1895     57500.0
1999    100000.0
2722     75000.0
3338    125000.0
Name: salary_usd, dtype: float64

In [ ]:
no_degree_jobs['salary_usd'] = (
    no_degree_jobs['salary_usd']
    .round()
    .astype('Int64')
)

no_degree_jobs.loc[no_degree_jobs['salary_usd'].notna(), 'salary_usd'].head(10)


157     117500
468     100500
887      90000
968      75000
999      55000
1343     60000
1895     57500
1999    100000
2722     75000
3338    125000
Name: salary_usd, dtype: Int64

In [82]:
# Function to put salary into tiers for better visualizations later
def categorize_salary(df: pd.DataFrame, salary_column: str) -> pd.DataFrame:
    """
    Categorizes salaries into income tiers: 'Low', 'Mid', or 'High' based on defined thresholds.

    Args:
        df (pd.DataFrame): The DataFrame containing a salary column as integers.
        salary_column (str): The name of the column with salary values (e.g., 'salary_usd').

    Returns:
        pd.DataFrame: The modified DataFrame with a new 'salary_tier' column.
    """
    bins = [0, 50000, 100000, float('inf')]
    labels = ['Low', 'Mid', 'High']
    df['salary_tier'] = pd.cut(df[salary_column], bins=bins, labels=labels, include_lowest=True)
    return df

In [87]:
no_degree_jobs = categorize_salary(no_degree_jobs, 'salary_usd')

In [86]:
no_degree_jobs.loc[no_degree_jobs['salary_usd'].notna(), 'salary_usd'].head(10)


157     117500
468     100500
887      90000
968      75000
999      55000
1343     60000
1895     57500
1999    100000
2722     75000
3338    125000
Name: salary_usd, dtype: Int64

In [88]:
no_degree_jobs.loc[
    no_degree_jobs['salary_usd'].notna(),
    ['salary_usd', 'salary_tier']
].head()

,salary_usd,salary_tier
157,117500,High
468,100500,High
887,90000,Mid
968,75000,Mid
999,55000,Mid
